# M7. 개선 실험: 통계 문서를 인덱스에 추가 (07_m7_개선실험)

**한 줄 요약** M6 평가에서 사고사망 통계 질문 16문항의 Hit@5가 0.00이었습니다. 원인은 검색이 안 되는 것이 아니라 **인덱스에 통계 문서가 아예 없기 때문**입니다.
이 노트북은 통계 문서를 추가한 새 컬렉션(`sif_v2`)을 만들고, **같은 골든 셋 100문항**으로 개선 전(`sif`)과 후(`sif_v2`)를 비교합니다.

| 항목 | 내용 |
|---|---|
| 가설 | 규모별 사고사망자수(파일 3)를 규모별 문서 10건으로 만들어 넣으면 통계 질문이 검색되고, 사례 질문 성능은 떨어지지 않는다 |
| 바꾸는 것 | 인덱스에 문서 추가 (그 외 임베딩 모델, k, 프롬프트는 그대로) |
| 비교 방법 | 같은 질문 · 같은 평가 함수로 `sif` vs `sif_v2` (1차), 결과를 진단한 뒤 `sif_v3` (2차) |
| 보는 지표 | Hit@1/3/5, MRR@5, 통계 답변 정확도, 사례 성능 하락 여부 |

- 기존 `sif`는 건드리지 않습니다. 사례 6,032건의 **임베딩을 복사**해서 새 컬렉션을 만들기 때문에 임베딩 비용이 거의 들지 않습니다(통계 문서 10건 + 질문 임베딩뿐).
- 사전 준비: 04 노트북으로 `data/chroma`가 있어야 하고, 06 노트북으로 `data/golden/natural_questions.csv`가 만들어져 있어야 합니다.

## 필요 패키지
**pandas, chromadb, openai, python-dotenv** 를 사용합니다(06과 동일). `ModuleNotFoundError`가 나면 설치 후 커널을 재시작하세요.
```
uv add pandas chromadb openai python-dotenv
```

## 0. 준비

In [29]:
import os, re
from pathlib import Path
import numpy as np
import pandas as pd
import chromadb
from dotenv import load_dotenv
from openai import OpenAI

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 120)

ROOT = Path.cwd()
while not (ROOT / "data" / "chroma").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data" / "chroma").exists(), "data/chroma가 없습니다. 04 노트북을 먼저 실행하세요."

load_dotenv(ROOT / ".env")
assert os.getenv("OPENAI_API_KEY"), f"{ROOT / '.env'} 에 OPENAI_API_KEY=키값 을 넣고 커널을 재시작하세요."
client = OpenAI(timeout=60, max_retries=2)

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o-mini")
K = 5
OLD, NEW = "sif", "sif_v2"

chroma = chromadb.PersistentClient(path=str(ROOT / "data" / "chroma"))
old = chroma.get_collection(OLD)
GOLD = ROOT / "data" / "golden" / "golden_set_100.csv"
NAT = ROOT / "data" / "golden" / "natural_questions.csv"
A = ROOT / "data" / "analysis"; A.mkdir(parents=True, exist_ok=True)
assert GOLD.exists(), f"{GOLD} 가 없습니다."
assert NAT.exists(), f"{NAT} 가 없습니다. 06 노트북을 먼저 끝까지 실행하세요."
print("개선 전 컬렉션", OLD, "문서 수:", old.count())

개선 전 컬렉션 sif 문서 수: 6032


## 1. 통계 문서 만들기
파일 3(`3_규모별_사고사망자수.csv`)은 **규모 10행 × 연도 22열(2004~2025)** 표입니다.
표 한 행을 문서 1건으로 바꿉니다. 검색(임베딩)은 문장을 대상으로 하므로 숫자를 문장으로 풀어 씁니다.

- 문서 ID: `통계-<규모>` (골든 셋의 정답 ID와 같은 규칙)
- 연도별 값을 `2025년 사고사망자수는 354명.` 형태의 문장과 `2025년 354명` 형태의 요약으로 모두 넣어, 질문 표현이 달라도 걸리게 합니다.
- 파일의 규모 표기(`5-9인`)와 사람이 묻는 표기(`5인 ~ 9인`)가 달라서, 규모를 **두 가지 표기로** 같이 적습니다.
- 최대·최소·증감 같은 요약 문장도 한 줄 넣습니다. (요약 숫자는 표에서 계산한 값입니다.)

In [30]:
def find3():
    cands = [p for p in (ROOT / "data" / "processed").glob("*.csv") if p.name.startswith("3_")]
    assert len(cands) == 1, f"3번 파일을 찾지 못했습니다: {cands}"
    return cands[0]

f3 = find3()
try:
    t3 = pd.read_csv(f3, encoding="utf-8-sig")
except UnicodeDecodeError:
    t3 = pd.read_csv(f3, encoding="cp949")
print(f3.name, t3.shape)

def size_alias(size):
    """파일의 규모 표기를 사람이 질문하는 표기로 바꾼다. 5-9인 -> 5인 ~ 9인, 5인미만 -> 5인 미만, 1000인이상 -> 1,000인 이상"""
    s = re.sub(r"[\s,]", "", str(size))
    fmt = lambda n: f"{int(n):,}"
    m = re.fullmatch(r"(\d+)[-~](\d+)인", s)
    if m: return f"{fmt(m.group(1))}인 ~ {fmt(m.group(2))}인"
    m = re.fullmatch(r"(\d+)인(미만|이상)", s)
    if m: return f"{fmt(m.group(1))}인 {m.group(2)}"
    return str(size).strip()

def build_stat_docs(df):
    years = [c for c in df.columns if re.fullmatch(r"\d{4}년", str(c))]
    ids, docs, metas = [], [], []
    for _, r in df.iterrows():
        size = str(r["구분"]).strip()
        vals = {}
        for c in years:
            v = pd.to_numeric(str(r[c]).replace(",", ""), errors="coerce")
            if pd.notna(v):
                vals[int(c[:4])] = int(v)
        y0, y1 = min(vals), max(vals)
        mx, mn = max(vals, key=vals.get), min(vals, key=vals.get)
        chg = (vals[y1] - vals[y0]) / vals[y0] * 100
        sent = " ".join(f"{y}년 사고사망자수는 {v}명." for y, v in vals.items())
        compact = ", ".join(f"{y}년 {v}명" for y, v in vals.items())
        alias = size_alias(size)
        doc = (f"[통계] 규모별 사고사망자수\n[규모] {size} 사업장 ({alias})\n"
               f"[대상] {alias} 사업장의 연도별 사고사망자 수\n"
               f"[연도별 사고사망자수] {sent}\n[연도별 요약] {compact}\n"
               f"[추이] {y0}~{y1}년 중 가장 많았던 해는 {mx}년({vals[mx]}명), 가장 적었던 해는 {mn}년({vals[mn]}명)이다. "
               f"{y0}년 대비 {y1}년은 {chg:+.1f}% 변했다.")
        ids.append("통계-" + re.sub(r"[\s,]", "", size))
        docs.append(doc)
        metas.append({"업종구분": "통계", "중분류": "규모별 사고사망자수", "기인물": "", "재해종류": "", "재해연도": 0})
    return ids, docs, metas

s_ids, s_docs, s_metas = build_stat_docs(t3)
assert len(set(s_ids)) == len(s_ids), "통계 문서 ID가 중복됩니다."
print("통계 문서 수:", len(s_ids), "| ID:", s_ids)
print()
print(s_docs[0])

3_규모별_사고사망자수.csv (10, 23)
통계 문서 수: 10 | ID: ['통계-5인미만', '통계-5-9인', '통계-10-19인', '통계-20-29인', '통계-30-49인', '통계-50-99인', '통계-100-299인', '통계-300-499인', '통계-500-999인', '통계-1000인이상']

[통계] 규모별 사고사망자수
[규모] 5인미만 사업장 (5인 미만)
[대상] 5인 미만 사업장의 연도별 사고사망자 수
[연도별 사고사망자수] 2004년 사고사망자수는 292명. 2005년 사고사망자수는 267명. 2006년 사고사망자수는 278명. 2007년 사고사망자수는 310명. 2008년 사고사망자수는 389명. 2009년 사고사망자수는 376명. 2010년 사고사망자수는 315명. 2011년 사고사망자수는 370명. 2012년 사고사망자수는 326명. 2013년 사고사망자수는 336명. 2014년 사고사망자수는 310명. 2015년 사고사망자수는 304명. 2016년 사고사망자수는 301명. 2017년 사고사망자수는 292명. 2018년 사고사망자수는 322명. 2019년 사고사망자수는 301명. 2020년 사고사망자수는 312명. 2021년 사고사망자수는 318명. 2022년 사고사망자수는 342명. 2023년 사고사망자수는 278명. 2024년 사고사망자수는 309명. 2025년 사고사망자수는 354명.
[연도별 요약] 2004년 292명, 2005년 267명, 2006년 278명, 2007년 310명, 2008년 389명, 2009년 376명, 2010년 315명, 2011년 370명, 2012년 326명, 2013년 336명, 2014년 310명, 2015년 304명, 2016년 301명, 2017년 292명, 2018년 322명, 2019년 301명, 2020년 312명, 2021년 318명, 2022년 342명, 2023년 278명, 2024년 309명, 2025년 354명
[추이] 2004~2025년 중 가장 많았던 해는 20

## 2. 새 컬렉션 `sif_v2` 만들기
1. `sif`의 문서·메타데이터·임베딩을 그대로 복사한다 (500건씩).
2. 통계 문서 10건을 임베딩해서 추가한다.

이미 `sif_v2`가 있고 문서 수가 맞으면 **그대로 재사용**합니다(다시 실행해도 결과가 같게). 처음부터 다시 만들려면 `REBUILD = True`로 바꾸세요.
컬렉션을 새로 만들 때는 `search_ef=100`(검색 폭)을 씁니다. ChromaDB의 검색은 근사(ANN)라 폭이 좁으면 정답을 놓칠 수 있습니다(8-3절에서 확인).

In [31]:
def embed_many(texts, bs=100):
    out = []
    for s in range(0, len(texts), bs):
        r = client.embeddings.create(model=EMBED_MODEL, input=[t[:6000] for t in texts[s:s + bs]])
        out += [x.embedding for x in r.data]
    return out

REBUILD = False   # True로 바꾸면 sif_v2 / sif_v3를 지우고 처음부터 다시 만든다 (다시 만들 때마다 근사 검색 결과가 조금씩 달라질 수 있다)
N = old.count()

def build_version(name, ids, docs, metas):
    """sif에 (ids, docs, metas)를 더한 컬렉션을 만든다. 이미 있고 문서 수가 맞으면 그대로 재사용한다."""
    expected = N + len(ids)
    if not REBUILD:
        try:
            c = chroma.get_collection(name)
            if c.count() == expected:
                print(f"{name}: 기존 컬렉션 재사용 ({c.count()}건). 처음부터 다시 만들려면 REBUILD = True")
                return c
        except Exception:
            pass
    try:
        chroma.delete_collection(name)
    except Exception:
        pass
    try:   # 검색 폭(search_ef)을 넓혀 근사 검색 오차를 줄인다
        c = chroma.get_or_create_collection(
            name, metadata={"hnsw:space": "cosine", "hnsw:search_ef": 100, "hnsw:construction_ef": 200})
    except Exception as e:
        print("hnsw 옵션을 지원하지 않아 기본 설정으로 만듭니다:", type(e).__name__)
        c = chroma.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})
    for s in range(0, N, 500):
        g = old.get(limit=500, offset=s, include=["documents", "metadatas", "embeddings"])
        c.add(ids=g["ids"], documents=g["documents"], metadatas=g["metadatas"],
              embeddings=[[float(x) for x in e] for e in g["embeddings"]])
    c.add(ids=ids, documents=docs, metadatas=metas, embeddings=embed_many(docs))
    print(f"{name}: 새로 만듦 ({c.count()}건 = 사례 {N} + 추가 {len(ids)})")
    assert c.count() == expected
    return c

new = build_version(NEW, s_ids, s_docs, s_metas)

sif_v2: 기존 컬렉션 재사용 (6042건). 처음부터 다시 만들려면 REBUILD = True


## 3. 골든 셋과 평가 함수 (06과 같은 정의)
- **Hit@K**: 상위 K개 안에 정답 문서가 있는 문항 비율, **Precision@K**: 상위 K개 중 관련 문서(정답 + 같은 기인물 사례)의 비율, **MRR@5**: 정답 순위 역수의 평균.
- 추가로 **기인물 Hit@5**(참고 지표)를 봅니다. 정답 사례 1건만 인정하면 "비슷한 사례를 가져왔는데 틀린 것"으로 처리되기 때문에, 정답 사례와 **기인물이 같은 사례가 상위 5개에 있는가**도 같이 봅니다. 통계 문항에는 적용하지 않습니다.
- 개선 전후 모두 **같은 질문 임베딩**으로 검색해서 질문 쪽 차이가 끼어들지 않게 합니다.

In [32]:
gold = pd.read_csv(GOLD, encoding="utf-8-sig")
gold["유형"] = gold["분야"].map(lambda s: "통계" if "통계" in s else "사례")
nat = pd.read_csv(NAT, encoding="utf-8-sig")
gold_nat = gold[gold["유형"] == "사례"].merge(nat, on="ID")

# 정답 사례의 기인물 (참고 지표용)
case_ids = list(gold.loc[gold["유형"] == "사례", "정답ID"])
gm = old.get(ids=case_ids, include=["metadatas"])
gold_cause = {i: m.get("기인물", "") for i, m in zip(gm["ids"], gm["metadatas"])}

def evaluate(df, embs, c, k=K):
    res = c.query(query_embeddings=embs, n_results=k, include=["metadatas", "distances"])
    rows = []
    for (_, r), ids, metas, dist in zip(df.iterrows(), res["ids"], res["metadatas"], res["distances"]):
        rank = ids.index(r["정답ID"]) + 1 if r["정답ID"] in ids else 0
        cause = gold_cause.get(r["정답ID"], "")
        same = (any(m.get("기인물", "") == cause for m in metas) if cause else np.nan) if r["유형"] == "사례" else np.nan
        rel = [(i == r["정답ID"]) or (bool(cause) and r["유형"] == "사례" and m.get("기인물", "") == cause) for i, m in zip(ids, metas)]
        rows.append({"ID": r["ID"], "유형": r["유형"], "분야": r["분야"], "정답ID": r["정답ID"],
                     "P@3": sum(rel[:3]) / 3, "P@5": sum(rel[:5]) / 5, "정답순위": rank, "정답유사도": round(1 - dist[ids.index(r["정답ID"])], 3) if rank else np.nan,
                     "1위ID": ids[0], "1위유사도": round(1 - dist[0], 3), "기인물일치": float(same) if same == same else np.nan})
    return pd.DataFrame(rows)

def metrics(res):
    return pd.Series({
        "문항수": len(res),
        "Hit@1": round((res["정답순위"] == 1).mean(), 3),
        "Hit@3": round(res["정답순위"].between(1, 3).mean(), 3),
        "Hit@5": round(res["정답순위"].between(1, 5).mean(), 3),
        "MRR@5": round(res["정답순위"].map(lambda x: 1 / x if x else 0).mean(), 3),
        "Precision@3": round(res["P@3"].mean(), 3),
        "Precision@5": round(res["P@5"].mean(), 3),
        "기인물Hit@5": round(res["기인물일치"].mean(), 3),
    })

E_all = embed_many(gold["질문"].tolist())
E_nat = embed_many(gold_nat["현장질문"].tolist())
print("질문 임베딩 완료:", len(E_all), "+", len(E_nat))

질문 임베딩 완료: 100 + 84


## 4. 개선 전후 비교

In [33]:
before_all = evaluate(gold, E_all, old)
after_all = evaluate(gold, E_all, new)
q_nat = gold_nat.copy(); q_nat["질문"] = q_nat["현장질문"]
before_nat = evaluate(q_nat, E_nat, old)
after_nat = evaluate(q_nat, E_nat, new)

sets = [
    ("원본 질문 · 통계 16", before_all[before_all["유형"] == "통계"], after_all[after_all["유형"] == "통계"]),
    ("원본 질문 · 사례 84", before_all[before_all["유형"] == "사례"], after_all[after_all["유형"] == "사례"]),
    ("현장 말투 질문 · 사례 84", before_nat, after_nat),
    ("원본 질문 · 전체 100", before_all, after_all),
]
rows = []
for label, b, a in sets:
    mb, ma = metrics(b), metrics(a)
    for m in ["Hit@1", "Hit@3", "Hit@5", "Precision@5", "MRR@5", "기인물Hit@5"]:
        rows.append({"질문 세트": label, "지표": m, "개선 전(sif)": mb[m], "개선 후(sif_v2)": ma[m],
                     "변화": round(ma[m] - mb[m], 3) if mb[m] == mb[m] else np.nan})
cmp = pd.DataFrame(rows)
display(cmp.pivot(index="지표", columns="질문 세트", values="변화").loc[["Hit@1", "Hit@3", "Hit@5", "Precision@5", "MRR@5", "기인물Hit@5"]])
display(cmp)
cmp.to_csv(A / "m7_전후비교.csv", index=False, encoding="utf-8-sig")
after_all.to_csv(A / "m7_평가결과_원본질문.csv", index=False, encoding="utf-8-sig")
after_nat.to_csv(A / "m7_평가결과_현장질문.csv", index=False, encoding="utf-8-sig")

질문 세트,원본 질문 · 사례 84,원본 질문 · 전체 100,원본 질문 · 통계 16,현장 말투 질문 · 사례 84
지표,,,,
Hit@1,-0.024,0.030,0.312,0.0
Hit@3,-0.035,0.040,0.438,0.0
Hit@5,-0.012,0.080,0.562,0.0
Precision@5,-0.002,0.016,0.112,0.0
MRR@5,-0.022,0.044,0.396,0.0
기인물Hit@5,-0.026,-0.026,NaN,0.0


,질문 세트,지표,개선 전(sif),개선 후(sif_v2),변화
0,원본 질문 · 통계 16,Hit@1,0.000,0.312,0.312
1,원본 질문 · 통계 16,Hit@3,0.000,0.438,0.438
2,원본 질문 · 통계 16,Hit@5,0.000,0.562,0.562
3,원본 질문 · 통계 16,Precision@5,0.000,0.112,0.112
4,원본 질문 · 통계 16,MRR@5,0.000,0.396,0.396
5,원본 질문 · 통계 16,기인물Hit@5,NaN,NaN,NaN
6,원본 질문 · 사례 84,Hit@1,0.762,0.738,-0.024
7,원본 질문 · 사례 84,Hit@3,0.833,0.798,-0.035
8,원본 질문 · 사례 84,Hit@5,0.845,0.833,-0.012
9,원본 질문 · 사례 84,Precision@5,0.390,0.388,-0.002


### 4-1. 사례 성능이 떨어졌는가 (부작용 점검)
통계 문서는 숫자와 연도가 가득한 긴 문서입니다. 사례 질문의 상위 5개를 **통계 문서가 차지하면** 사례 성능이 나빠질 수 있습니다.
- 사례 질문 중 상위 5개에 통계 문서가 들어온 건수를 세고,
- 개선 전에는 정답이었는데 개선 후에 놓친 문항이 있는지 확인합니다.

In [34]:
res = new.query(query_embeddings=E_all, n_results=K)
case_mask = (gold["유형"] == "사례").values
polluted = [sum(i.startswith("통계-") for i in ids) > 0 for ids, m in zip(res["ids"], case_mask) if m]
print(f"원본 사례 질문 84개 중 상위 5개에 통계 문서가 섞인 질문: {sum(polluted)}개")

res2 = new.query(query_embeddings=E_nat, n_results=K)
print(f"현장 말투 사례 질문 {len(E_nat)}개 중 상위 5개에 통계 문서가 섞인 질문: {sum(any(i.startswith('통계-') for i in ids) for ids in res2['ids'])}개")

lost = before_all.merge(after_all, on="ID", suffixes=("_전", "_후"))
lost = lost[(lost["유형_전"] == "사례") & lost["정답순위_전"].between(1, 5) & (lost["정답순위_후"] == 0)]
print("개선 전에는 5위 안이었다가 개선 후 5위 밖으로 밀린 사례 문항:", len(lost))
display(lost[["ID", "정답ID_전", "정답순위_전", "정답순위_후", "1위ID_후"]])

원본 사례 질문 84개 중 상위 5개에 통계 문서가 섞인 질문: 0개
현장 말투 사례 질문 84개 중 상위 5개에 통계 문서가 섞인 질문: 0개
개선 전에는 5위 안이었다가 개선 후 5위 밖으로 밀린 사례 문항: 2


,ID,정답ID_전,정답순위_전,정답순위_후,1위ID_후
17,Q18,제조업등-1017,1,0,건설업-1519
38,Q39,제조업등-1776,1,0,건설업-3214


## 5. 통계 질문의 답변 정확도 (검색 → 생성까지)
검색에서 정답 문서를 찾더라도 LLM이 숫자를 정확히 읽는지는 별개입니다. 통계 16문항에 대해
- 상위 3개 문서를 근거로 답하게 하고,
- 골든 셋의 `정답확인문구`에 들어 있는 **정답 숫자**(예: 354명)가 답변에 들어 있는지 확인합니다.

개선 전 컬렉션은 통계 문서가 없어 근거 없음으로 답해야 정상입니다.

In [35]:
SYSTEM = ("당신은 산업안전 도우미입니다. 아래 [자료]에 있는 내용만 근거로 한국어로 짧게 답하세요. "
          "자료에 근거가 없으면 정확히 '제공된 사례에서 근거를 찾을 수 없습니다.'라고만 답하세요.")

def answer(q, qemb, c, k=3):
    r = c.query(query_embeddings=[qemb], n_results=k)
    ctx = "\n\n".join(f"[자료{i + 1}]\n{d}" for i, d in enumerate(r["documents"][0]))
    out = client.chat.completions.create(
        model=CHAT_MODEL, temperature=0,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"{ctx}\n\n질문: {q}"}],
    ).choices[0].message.content.strip()
    return out, r["ids"][0]

stat = gold[gold["유형"] == "통계"].copy()
stat_emb = [E_all[i] for i in stat.index]
expect = stat["정답확인문구"].map(lambda s: re.search(r"(\d[\d,]*)\s*명", s).group(1).replace(",", ""))

rows = []
for (idx, r), e, ex in zip(stat.iterrows(), stat_emb, expect):
    row = {"ID": r["ID"], "질문": r["질문"], "정답숫자": ex}
    for name, c in [("전", old), ("후", new)]:
        ans, ids3 = answer(r["질문"], e, c)
        row[f"답변_{name}"] = ans
        row[f"정답_{name}"] = ex in ans.replace(",", "")
    row["후_근거ID"] = ids3[0]
    rows.append(row)
ans_df = pd.DataFrame(rows)
print(f"통계 16문항 답변 정확도 — 개선 전: {ans_df['정답_전'].mean():.2f} / 개선 후: {ans_df['정답_후'].mean():.2f}")
display(ans_df[["ID", "질문", "정답숫자", "답변_후", "정답_후", "후_근거ID"]])
ans_df.to_csv(A / "m7_통계답변정확도.csv", index=False, encoding="utf-8-sig")

통계 16문항 답변 정확도 — 개선 전: 0.00 / 개선 후: 0.38


,ID,질문,정답숫자,답변_후,정답_후,후_근거ID
0,Q03,2025년 5인 미만 사업장의 사고사망자 수는 몇 명인가요?,354,제공된 사례에서 근거를 찾을 수 없습니다.,False,통계-500-999인
1,Q86,2025년 5인 ~ 9인 사업장의 사고사망자 수는 몇 명인가요?,107,제공된 사례에서 근거를 찾을 수 없습니다.,False,통계-500-999인
2,Q87,2025년 10인 ~ 19인 사업장의 사고사망자 수는 몇 명인가요?,113,2025년 10인 ~ 19인 사업장의 사고사망자 수는 113명입니다.,True,통계-20-29인
3,Q88,2025년 20인 ~ 29인 사업장의 사고사망자 수는 몇 명인가요?,49,제공된 사례에서 근거를 찾을 수 없습니다.,False,통계-5-9인
4,Q89,2025년 30인 ~ 49인 사업장의 사고사망자 수는 몇 명인가요?,63,제공된 사례에서 근거를 찾을 수 없습니다.,False,통계-5-9인
5,Q90,2025년 50인 ~ 99인 사업장의 사고사망자 수는 몇 명인가요?,47,제공된 사례에서 근거를 찾을 수 없습니다.,False,통계-5-9인
6,Q91,2025년 100인 ~ 299인 사업장의 사고사망자 수는 몇 명인가요?,74,제공된 사례에서 근거를 찾을 수 없습니다.,False,통계-20-29인
7,Q92,2025년 300인 ~ 499인 사업장의 사고사망자 수는 몇 명인가요?,12,제공된 사례에서 근거를 찾을 수 없습니다.,False,통계-5-9인
8,Q93,2025년 500인 ~ 999인 사업장의 사고사망자 수는 몇 명인가요?,19,제공된 사례에서 근거를 찾을 수 없습니다.,False,통계-5-9인
9,Q94,"2025년 1,000인 이상 사업장의 사고사망자 수는 몇 명인가요?",34,"2025년 1,000인 이상 사업장의 사고사망자 수는 34명입니다.",True,통계-1000인이상


## 6. RAG 체인의 `MIN_SIM`(0.40) 기준을 통과하는가
05의 RAG 체인은 상위 문서의 유사도가 `MIN_SIM = 0.40` 미만이면 "근거 없음"으로 답합니다.
통계 문서가 검색은 되는데 유사도가 0.40 미만이면 **실제 챗봇에서는 답하지 못합니다.** 정답 통계 문서의 유사도를 확인합니다.

In [36]:
MIN_SIM = 0.40
sa = after_all[after_all["유형"] == "통계"]
print("통계 질문의 정답 문서 유사도: 최소 %.3f / 중앙값 %.3f / 최대 %.3f" % (sa["정답유사도"].min(), sa["정답유사도"].median(), sa["정답유사도"].max()))
print(f"MIN_SIM {MIN_SIM} 이상인 통계 질문: {(sa['정답유사도'] >= MIN_SIM).sum()} / {len(sa)}")
ca = after_all[after_all["유형"] == "사례"]
print("(참고) 사례 질문의 정답 문서 유사도 중앙값: %.3f" % ca["정답유사도"].median())

통계 질문의 정답 문서 유사도: 최소 0.598 / 중앙값 0.627 / 최대 0.644
MIN_SIM 0.4 이상인 통계 질문: 9 / 16
(참고) 사례 질문의 정답 문서 유사도 중앙값: 0.708


## 7. 진단: 통계 Hit@5가 왜 0.6 정도에 그칠까
1차 결과(`sif_v2`)는 통계 Hit@5가 0.5~0.7 사이에서 **실행마다 흔들렸습니다**(같은 질문, 같은 문서인데 컬렉션을 다시 만들 때마다 0.56~0.69). **통계 문서가 10건뿐**이라 무작위로 5건을 골라도 Hit@5는 0.50이므로 이 정도는 좋은 결과가 아닙니다.
정답 문서와 다른 규모 문서의 유사도 차이가 얼마나 되는지 확인합니다.

In [37]:
sv2 = after_all[after_all["유형"] == "통계"]
print("정답 순위 분포 (0 = 5위 밖):", sv2["정답순위"].value_counts().sort_index().to_dict())
print("1위로 가장 많이 나온 문서:", sv2["1위ID"].value_counts().head(3).to_dict())
print("무작위로 통계 문서 5건을 고를 때 Hit@5 기대값: 0.50")

stat_pos = list(gold.index[gold["유형"] == "통계"])
r10 = new.query(query_embeddings=[E_all[i] for i in stat_pos], n_results=10, where={"업종구분": "통계"})
spread = [max(1 - d for d in ds) - min(1 - d for d in ds) for ds in r10["distances"]]
print("한 질문에서 통계 문서 10건 사이의 유사도 차이(최대-최소): 평균 %.3f" % np.mean(spread))
print("→ 통계 문서끼리 유사도 차이가 매우 작아 순위가 근소한 차이로 갈린다. 이런 경우 근사 검색은 순위를 안정적으로 가리지 못한다(8-3 참고). 문서 한 건에 22개 연도 숫자가 모두 들어 있어 문서끼리 비슷해진 것이 원인.")

정답 순위 분포 (0 = 5위 밖): {0: 7, 1: 5, 2: 1, 3: 1, 4: 2}
1위로 가장 많이 나온 문서: {'통계-5-9인': 6, '통계-500-999인': 3, '통계-20-29인': 2}
무작위로 통계 문서 5건을 고를 때 Hit@5 기대값: 0.50
한 질문에서 통계 문서 10건 사이의 유사도 차이(최대-최소): 평균 0.033
→ 통계 문서끼리 유사도 차이가 매우 작아 순위가 근소한 차이로 갈린다. 이런 경우 근사 검색은 순위를 안정적으로 가리지 못한다(8-3 참고). 문서 한 건에 22개 연도 숫자가 모두 들어 있어 문서끼리 비슷해진 것이 원인.


## 8. 2차 개선: 규모×연도 단위로 잘게 쪼개기 (`sif_v3`)
- 문서 1건 = **규모 1개 × 연도 1개** (10 × 22 = 220건). 문서 안의 숫자가 1개라서 질문의 연도·규모와 문서가 거의 1:1로 맞습니다.
- ID: `통계-<규모>-<연도>` (예: `통계-5-9인-2025`)
- 골든 셋의 통계 정답 ID도 같은 규칙으로 바꿔서 평가합니다. (`통계-5-9인` + 정답 문구의 연도 → `통계-5-9인-2025`) **연도까지 맞아야 정답**이라 1차보다 엄격한 기준입니다.
- 이것은 RAG에서 말하는 "청킹(문서를 어떤 단위로 자를 것인가)" 조정입니다.

In [38]:
def build_stat_year_docs(df):
    ids, docs, metas = [], [], []
    for _, r in df.iterrows():
        size = str(r["구분"]).strip()
        alias = size_alias(size)
        base = "통계-" + re.sub(r"[\s,]", "", size)
        for c in df.columns:
            if not re.fullmatch(r"\d{4}년", str(c)):
                continue
            v = pd.to_numeric(str(r[c]).replace(",", ""), errors="coerce")
            if pd.isna(v):
                continue
            y, v = int(c[:4]), int(v)
            ids.append(f"{base}-{y}")
            docs.append(f"[통계] 규모별 사고사망자수\n[규모] {size} 사업장 ({alias})\n[연도] {y}년\n"
                        f"{y}년 {alias} 사업장의 사고사망자수는 {v}명.\n[요약] {y}년 {v}명")
            metas.append({"업종구분": "통계", "중분류": "규모별 사고사망자수", "기인물": "", "재해종류": "", "재해연도": y})
    return ids, docs, metas

y_ids, y_docs, y_metas = build_stat_year_docs(t3)
assert len(set(y_ids)) == len(y_ids)
print("규모×연도 통계 문서 수:", len(y_ids))
print(y_docs[0])

NEW3 = "sif_v3"
new3 = build_version(NEW3, y_ids, y_docs, y_metas)

# 통계 정답 ID를 규모-연도 단위로 변환
gold_v3 = gold.copy()
m = gold_v3["유형"] == "통계"
gold_v3.loc[m, "정답ID"] = [f"{i}-{re.search(r'(\d{4})년', p).group(1)}"
                          for i, p in zip(gold_v3.loc[m, "정답ID"], gold_v3.loc[m, "정답확인문구"])]
assert len(new3.get(ids=list(gold_v3.loc[m, "정답ID"]))["ids"]) == m.sum(), "정답 ID가 sif_v3에 없습니다."
print("통계 정답 ID 변환 예:", list(gold_v3.loc[m, "정답ID"])[:3])

규모×연도 통계 문서 수: 220
[통계] 규모별 사고사망자수
[규모] 5인미만 사업장 (5인 미만)
[연도] 2004년
2004년 5인 미만 사업장의 사고사망자수는 292명.
[요약] 2004년 292명
sif_v3: 기존 컬렉션 재사용 (6252건). 처음부터 다시 만들려면 REBUILD = True
통계 정답 ID 변환 예: ['통계-5인미만-2025', '통계-5-9인-2025', '통계-10-19인-2025']


### 8-1. 3단계 비교: 개선 전 → v2(규모 단위) → v3(규모×연도 단위)

In [39]:
after_all3 = evaluate(gold_v3, E_all, new3)
after_nat3 = evaluate(q_nat, E_nat, new3)

sets3 = [
    ("원본 질문 · 통계 16", before_all[before_all["유형"] == "통계"], after_all[after_all["유형"] == "통계"], after_all3[after_all3["유형"] == "통계"]),
    ("원본 질문 · 사례 84", before_all[before_all["유형"] == "사례"], after_all[after_all["유형"] == "사례"], after_all3[after_all3["유형"] == "사례"]),
    ("현장 말투 질문 · 사례 84", before_nat, after_nat, after_nat3),
]
rows = []
for label, b, a2, a3 in sets3:
    mb, m2, m3 = metrics(b), metrics(a2), metrics(a3)
    for k_ in ["Hit@1", "Hit@3", "Hit@5", "Precision@5", "MRR@5"]:
        rows.append({"질문 세트": label, "지표": k_, "개선 전(sif)": mb[k_], "v2(sif_v2)": m2[k_], "v3(sif_v3)": m3[k_]})
cmp3 = pd.DataFrame(rows)
display(cmp3)
cmp3.to_csv(A / "m7_전후비교_v3.csv", index=False, encoding="utf-8-sig")
after_all3.to_csv(A / "m7_평가결과_v3_원본질문.csv", index=False, encoding="utf-8-sig")
print("※ 통계 정답 기준: 개선 전·v2는 규모 단위, v3는 규모+연도까지 맞아야 정답(더 엄격).")

,질문 세트,지표,개선 전(sif),v2(sif_v2),v3(sif_v3)
0,원본 질문 · 통계 16,Hit@1,0.000,0.312,0.875
1,원본 질문 · 통계 16,Hit@3,0.000,0.438,1.000
2,원본 질문 · 통계 16,Hit@5,0.000,0.562,1.000
3,원본 질문 · 통계 16,Precision@5,0.000,0.112,0.200
4,원본 질문 · 통계 16,MRR@5,0.000,0.396,0.938
5,원본 질문 · 사례 84,Hit@1,0.762,0.738,0.762
6,원본 질문 · 사례 84,Hit@3,0.833,0.798,0.821
7,원본 질문 · 사례 84,Hit@5,0.845,0.833,0.857
8,원본 질문 · 사례 84,Precision@5,0.390,0.388,0.393
9,원본 질문 · 사례 84,MRR@5,0.796,0.774,0.798


※ 통계 정답 기준: 개선 전·v2는 규모 단위, v3는 규모+연도까지 맞아야 정답(더 엄격).


### 8-2. v3 부작용 점검, 통계 답변 정확도, MIN_SIM

In [40]:
res3 = new3.query(query_embeddings=E_all, n_results=K)
poll3 = sum(any(i.startswith("통계-") for i in ids) for ids, mk in zip(res3["ids"], case_mask) if mk)
resn3 = new3.query(query_embeddings=E_nat, n_results=K)
polln3 = sum(any(i.startswith("통계-") for i in ids) for ids in resn3["ids"])
print(f"사례 질문 상위 5개에 통계 문서가 섞인 건수 — 원본 질문: {poll3}/84, 현장 말투 질문: {polln3}/84")

acc3 = []
for (idx, r), e in zip(stat.iterrows(), stat_emb):
    ans, ids3 = answer(r["질문"], e, new3)
    ex = re.search(r"(\d[\d,]*)\s*명", r["정답확인문구"]).group(1).replace(",", "")
    acc3.append((ans, ex in ans.replace(",", "")))
ans_df["답변_v3"] = [a for a, _ in acc3]
ans_df["정답_v3"] = [c for _, c in acc3]
print(f"통계 16문항 답변 정확도 — 개선 전 {ans_df['정답_전'].mean():.2f} / v2 {ans_df['정답_후'].mean():.2f} / v3 {ans_df['정답_v3'].mean():.2f}")
display(ans_df[~ans_df["정답_v3"]][["ID", "질문", "정답숫자", "답변_v3"]])
ans_df.to_csv(A / "m7_통계답변정확도.csv", index=False, encoding="utf-8-sig")

sa3 = after_all3[after_all3["유형"] == "통계"]
print("v3 통계 정답 문서 유사도: 최소 %.3f / 중앙값 %.3f" % (sa3["정답유사도"].min(), sa3["정답유사도"].median()))
print(f"MIN_SIM {MIN_SIM} 이상: {(sa3['정답유사도'] >= MIN_SIM).sum()} / {len(sa3)}")

사례 질문 상위 5개에 통계 문서가 섞인 건수 — 원본 질문: 0/84, 현장 말투 질문: 0/84
통계 16문항 답변 정확도 — 개선 전 0.00 / v2 0.38 / v3 1.00


,ID,질문,정답숫자,답변_v3


v3 통계 정답 문서 유사도: 최소 0.749 / 중앙값 0.780
MIN_SIM 0.4 이상: 16 / 16


### 8-3. 근사 검색 점검: Chroma 결과가 정확한 검색과 얼마나 같은가
ChromaDB는 속도를 위해 **근사 검색(HNSW)** 을 씁니다. 그래서 컬렉션을 새로 만들 때마다 정답 문서를 한두 건 놓칠 수 있습니다.
(실제로 첫 실행에서 `sif_v3`의 사례 Hit@5가 2문항 내려갔고, 정확 검색으로 보면 두 문항 모두 개선 전과 같은 순위였습니다.)

벡터 전체를 numpy로 직접 비교하는 **정확 검색**과 비교해서
- 상위 5개가 얼마나 일치하는지(재현율),
- 정확 검색 기준 Hit@5가 어떻게 나오는지
를 확인합니다. 정확 검색 기준은 인덱스 오차가 없으니 개선 전/후를 가장 공정하게 비교할 수 있습니다.

In [41]:
def exact_topk(c, E, k=K):
    G = c.get(include=["embeddings"])
    ids_ = G["ids"]
    M = np.array(G["embeddings"], dtype=np.float32); M /= np.linalg.norm(M, axis=1, keepdims=True)
    Q = np.array(E, dtype=np.float32); Q /= np.linalg.norm(Q, axis=1, keepdims=True)
    S = Q @ M.T
    return [[ids_[j] for j in np.argsort(-row)[:k]] for row in S]

rows = []
for name, c, gd in [("개선 전(sif)", old, gold), ("v2(sif_v2)", new, gold), ("v3(sif_v3)", new3, gold_v3)]:
    approx = c.query(query_embeddings=E_all, n_results=K)["ids"]
    exact = exact_topk(c, E_all)
    recall = np.mean([len(set(a) & set(e)) / K for a, e in zip(approx, exact)])
    truth = gd["정답ID"].tolist()
    for typ in ["사례", "통계"]:
        mk = (gd["유형"] == typ).values
        h_ap = np.mean([t in a for t, a, m_ in zip(truth, approx, mk) if m_])
        h_ex = np.mean([t in e for t, e, m_ in zip(truth, exact, mk) if m_])
        rows.append({"컬렉션": name, "유형": typ, "근사 재현율(상위5)": round(recall, 3),
                     "Hit@5 (Chroma 근사)": round(h_ap, 3), "Hit@5 (정확 검색)": round(h_ex, 3)})
exact_cmp = pd.DataFrame(rows)
display(exact_cmp)
exact_cmp.to_csv(A / "m7_정확검색비교.csv", index=False, encoding="utf-8-sig")
print("※ 재현율이 1.0이 아니면 Chroma 근사 검색이 정확 검색과 다른 문서를 돌려준 질문이 있다는 뜻입니다.")

,컬렉션,유형,근사 재현율(상위5),Hit@5 (Chroma 근사),Hit@5 (정확 검색)
0,개선 전(sif),사례,0.966,0.845,0.869
1,개선 전(sif),통계,0.966,0.000,0.000
2,v2(sif_v2),사례,0.942,0.833,0.869
3,v2(sif_v2),통계,0.942,0.562,0.875
4,v3(sif_v3),사례,0.992,0.857,0.869
5,v3(sif_v3),통계,0.992,1.000,1.000


※ 재현율이 1.0이 아니면 Chroma 근사 검색이 정확 검색과 다른 문서를 돌려준 질문이 있다는 뜻입니다.


## 9. 요약

In [42]:
g3 = lambda label, m: cmp3[(cmp3["질문 세트"] == label) & (cmp3["지표"] == m)].iloc[0]
for label in ["원본 질문 · 통계 16", "원본 질문 · 사례 84", "현장 말투 질문 · 사례 84"]:
    r_ = g3(label, "Hit@5")
    print(f"[{label} Hit@5] {r_['개선 전(sif)']:.2f} → v2 {r_['v2(sif_v2)']:.2f} → v3 {r_['v3(sif_v3)']:.2f}")
print(f"[통계 답변 정확도] {ans_df['정답_전'].mean():.2f} → v2 {ans_df['정답_후'].mean():.2f} → v3 {ans_df['정답_v3'].mean():.2f}")
c_b, c_3 = g3("원본 질문 · 사례 84", "Hit@5")["개선 전(sif)"], g3("원본 질문 · 사례 84", "Hit@5")["v3(sif_v3)"]
n_b, n_3 = g3("현장 말투 질문 · 사례 84", "Hit@5")["개선 전(sif)"], g3("현장 말투 질문 · 사례 84", "Hit@5")["v3(sif_v3)"]
if c_3 >= c_b - 0.03 and n_3 >= n_b - 0.03:
    print("→ 사례 검색 성능은 유지됐다 (Chroma 근사 검색 기준).")
else:
    print("→ Chroma 근사 검색 기준으로 사례 성능이 내려갔다. 8-2(통계 문서 혼입 건수)와 8-3(정확 검색 기준)을 함께 보고 판단할 것.")
print()
best = "sif_v3" if ans_df["정답_v3"].mean() > ans_df["정답_후"].mean() else "sif_v2"
print(f"통계 답변 정확도가 더 높은 쪽: {best}")
ec = exact_cmp[exact_cmp["유형"] == "사례"].set_index("컬렉션")
print("[사례 Hit@5, 정확 검색 기준] " + " / ".join(f"{k_} {v:.3f}" for k_, v in ec["Hit@5 (정확 검색)"].items()))
print(f"적용 방법: 05 RAG 체인과 M8 앱의 COLLECTION 이름을 'sif' → '{best}'로 바꾼다.")

[원본 질문 · 통계 16 Hit@5] 0.00 → v2 0.56 → v3 1.00
[원본 질문 · 사례 84 Hit@5] 0.84 → v2 0.83 → v3 0.86
[현장 말투 질문 · 사례 84 Hit@5] 0.26 → v2 0.26 → v3 0.26
[통계 답변 정확도] 0.00 → v2 0.38 → v3 1.00
→ 사례 검색 성능은 유지됐다 (Chroma 근사 검색 기준).

통계 답변 정확도가 더 높은 쪽: sif_v3
[사례 Hit@5, 정확 검색 기준] 개선 전(sif) 0.869 / v2(sif_v2) 0.869 / v3(sif_v3) 0.869
적용 방법: 05 RAG 체인과 M8 앱의 COLLECTION 이름을 'sif' → 'sif_v3'로 바꾼다.


## 정리
- **1차 가설**: 통계 질문이 안 되는 이유는 인덱스에 통계 문서가 없어서다 → 규모별 통계 문서 10건을 추가(`sif_v2`) → Hit@5 0.00에서 올랐지만 실행마다 0.56~0.69로 흔들렸다(정확 검색 기준 0.875).
- **진단**: 규모별 문서 10건의 유사도 차이가 약 0.03으로 매우 작아(near-tie) 근사 검색이 순위를 안정적으로 가리지 못했다. 문서 한 건에 22개 연도 숫자가 모두 들어 있어 문서끼리 비슷해진 것이 원인이고, 긴 문서에서 LLM이 숫자를 놓치기도 했다(답변 정확도 0.3~0.4).
- **2차 개선**: 문서를 규모×연도 단위(220건)로 쪼갰다(`sif_v3`) = 청킹 단위 조정. 평가 기준도 연도까지 맞아야 정답으로 더 엄격하게 바꿨다.
- **검증**: 같은 골든 셋 · 같은 평가 함수. 사례 성능 하락 여부, 통계 답변 정확도, `MIN_SIM` 통과 여부, 근사 검색 대 정확 검색 비교(8-3)까지 확인.
- **발견**: 컬렉션을 새로 만들면 Chroma의 근사 검색이 정답을 한두 건 놓칠 수 있다(첫 실행에서 사례 Hit@5 −2문항, 정확 검색으로는 개선 전과 동일). 그래서 정확 검색 기준을 함께 보고하고, 컬렉션을 `search_ef`를 넓혀 만든다.
- **확인할 점**: 표 형태 데이터를 "행 하나에 연도 전체"로 넣을 때와 "숫자 하나 = 문서 하나"로 넣을 때의 차이를 8-1 비교표와 통계 답변 정확도로 확인한다.
- **한계**: 통계는 규모별 사망자수 한 가지뿐이다. 업종별·재해유형별 통계 질문은 답할 수 없다. 현장 말투 질문의 낮은 사례 검색 성능(Hit@5 0.26)은 이번에 해결하지 못했다(기인물 필터 등 후속 과제).